# Malaria Tweet NLP Pipeline
### Nigerian Twitter Infodemiology — BERTweet-based 5-Class Classifier

This notebook implements the full pipeline from the implementation plan:
- **Phase 1** – Dataset prep, BERTweet fine-tuning, SVM/LR baselines, evaluation  
- **Phase 2** – Streamlit dashboard (single tweet + batch CSV)  
- **Phase 3** – Real-time geospatial surveillance (ntscraper + Folium)

## Setup: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install Dependencies

In [ ]:
!pip install -q transformers[torch] datasets accelerate scikit-learn evaluate emoji==0.6.0
!pip install -q streamlit plotly pandas ntscraper folium streamlit-folium

## Phase 0: Load & Deduplicate Raw Data

Load the raw scraped JSONL, deduplicate on tweet text, and persist the cleaned JSON for all downstream steps.

In [ ]:
import pandas as pd
import os

UNIQUE_FILE = "/content/drive/MyDrive/NLP_twitter/unique_malaria_tweets.json"
SOURCE_FILE  = "/content/drive/MyDrive/NLP_twitter/malaria_tweets.jsonl"

if os.path.exists(UNIQUE_FILE):
    print(f"--- {UNIQUE_FILE} exists. Loading unique data. ---")
    df_unique = pd.read_json(UNIQUE_FILE)
else:
    print("--- Cleaning and deduplicating... ---")
    df = pd.read_json(SOURCE_FILE, lines=True)
    df = df[["text", "created_at", "author", "views", "favorite_count", "is_reply"]].copy()
    df.rename(columns={"created_at": "date", "author": "user", "favorite_count": "likes"}, inplace=True)
    df['text'] = df['text'].str.strip('"').str.strip()
    df_unique = df.drop_duplicates(subset=['text']).copy()
    df_unique["likes"] = pd.to_numeric(df_unique["likes"], errors='coerce').fillna(0)
    df_unique.to_json(UNIQUE_FILE, orient='records', indent=4, force_ascii=False)
    print(f"--- Saved {len(df_unique)} unique tweets to {UNIQUE_FILE} ---")

print(df_unique.head())

---
## Phase 1: Model Fine-Tuning — The "Engine"

### Step 1: Define 5-Class Taxonomy & Remap Labels

The pipeline uses a **5-class taxonomy**:

| ID | Label |
|----|-------|
| 0 | Symptoms & Burden |
| 1 | Treatment & Health System |
| 2 | Prevention & Awareness |
| 3 | Misinformation |
| 4 | Irrelevant |

> **Note:** Labels are 0-indexed here to match PyTorch's `CrossEntropyLoss` convention.  
> If your JSON stores 1-based integer keys, the mapping below handles the conversion.

In [ ]:
import pandas as pd
import re

# ── 5-class taxonomy (0-indexed for PyTorch) ────────────────────────────────
id2label = {0: "Symptoms & Burden",
            1: "Treatment & Health System",
            2: "Prevention & Awareness",
            3: "Misinformation",
            4: "Irrelevant"}
label2id = {v: k for k, v in id2label.items()}
NUM_LABELS = len(id2label)

# ── Load the deduplicated dataset ────────────────────────────────────────────
FILE_PATH = "/content/drive/MyDrive/NLP_twitter/unique_malaria_tweets.json"
df = pd.read_json(FILE_PATH)

# ── Map old 7-class string labels → new 5-class integer labels ───────────────
# (Only needed if the JSON was produced by the zero-shot BART classifier.
#  If 'label' already contains correct 0-4 integers, skip this block.)
if 'ai_label' in df.columns and 'label' not in df.columns:
    old_to_new = {
        "Symptoms":           0,   # Symptoms & Burden
        "Personal Experience":0,   # Symptoms & Burden
        "Treatment":          1,   # Treatment & Health System
        "Health System":      1,   # Treatment & Health System
        "Prevention":         2,   # Prevention & Awareness
        "Misinformation":     3,   # Misinformation
        "Irrelevant":         4,   # Irrelevant
    }
    df['label'] = df['ai_label'].map(old_to_new)
elif 'label' in df.columns:
    # If labels are already 1-based integers, convert to 0-based
    if df['label'].min() == 1:
        df['label'] = df['label'] - 1

df['label'] = df['label'].astype(int)

print("Label distribution (5-class):")
print(df['label'].value_counts().sort_index().rename(id2label))

### Step 2: BERTweet-Specific Preprocessing

BERTweet's training corpus used specific normalisation:
- `@mentions` → `@USER`  
- URLs → `HTTPURL`  
- **No lowercasing** — BERTweet is case-sensitive

In [ ]:
def preprocess_tweet(text):
    """Normalise tweet text for BERTweet input."""
    text = re.sub(r'http\S+', 'HTTPURL', str(text))
    text = re.sub(r'@\w+', '@USER', text)
    return text

df['text'] = df['text'].apply(preprocess_tweet)
print("Preprocessing done. Example:")
print(df['text'].iloc[0])

### Step 3: Stratified 80 / 10 / 10 Train-Val-Test Split

Stratify on label to prevent class-imbalance skew — "Symptoms & Burden" is expected to dominate.

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

print(f"Train : {len(train_df)}  |  Val : {len(val_df)}  |  Test : {len(test_df)}")
print("\nClass distribution — Train:")
print(train_df['label'].value_counts().sort_index().rename(id2label))

### Step 4: Compute Class Weights for Imbalance Handling

Weights are passed to `nn.CrossEntropyLoss` during fine-tuning instead of relying on oversampling.

In [ ]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

classes = np.arange(NUM_LABELS)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_df['label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", dict(zip([id2label[i] for i in classes], class_weights.round(3))))

### Step 5: Tokenise with BERTweet AutoTokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", normalization=True)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_dict(train_df[['text', 'label']]).map(tokenize_fn, batched=True)
val_dataset   = Dataset.from_dict(val_df[['text', 'label']]).map(tokenize_fn, batched=True)
test_dataset  = Dataset.from_dict(test_df[['text', 'label']]).map(tokenize_fn, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenisation complete.")

### Step 6: Fine-Tune BERTweet (Weighted Loss, Best by Macro F1)

Key config: LR `2e-5`, batch `16`, epochs `5`, warmup `0.1`, save best checkpoint on validation macro F1.

In [ ]:
import evaluate
import numpy as np
from transformers import (AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)

model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/bertweet-base",
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

# ── Weighted loss to fight class imbalance ───────────────────────────────────
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=preds, references=labels, average="macro")

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    push_to_hub=False,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

### Step 7: SVM & Logistic Regression Baselines (TF-IDF)

Required by RQ3. Train on the same preprocessed text splits, evaluate with macro F1 for comparison.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['text'])
X_val   = vectorizer.transform(val_df['text'])
X_test  = vectorizer.transform(test_df['text'])
y_train, y_val, y_test = train_df['label'], val_df['label'], test_df['label']

# SVM
svm = LinearSVC(class_weight='balanced', max_iter=2000)
svm.fit(X_train, y_train)
print("=== SVM — Test Set ===")
print(classification_report(y_test, svm.predict(X_test),
                             target_names=list(id2label.values())))

# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, solver='lbfgs',
                         multi_class='multinomial', C=1.0)
lr.fit(X_train, y_train)
print("=== Logistic Regression — Test Set ===")
print(classification_report(y_test, lr.predict(X_test),
                             target_names=list(id2label.values())))

### Step 8: Evaluate Fine-Tuned BERTweet

Generate per-class F1 report and a confusion matrix. Focus on:
- "Misinformation" vs "Treatment & Health System" overlap  
- "Irrelevant" precision (leaky irrelevant pollutes public health signal)

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# Get predictions on the test set
preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print("=== BERTweet — Test Set ===")
print(classification_report(y_true, y_pred, target_names=list(id2label.values())))

# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=list(id2label.values()),
    ax=ax, colorbar=False, xticks_rotation=45
)
plt.title("BERTweet Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

### Step 9: Save Fine-Tuned Model & Tokenizer to Drive

In [ ]:
import os

MODEL_SAVE_PATH = "/content/drive/MyDrive/NLP_twitter/saved_bertweet"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print(f"Model and tokenizer saved to: {MODEL_SAVE_PATH}")

---
## Exploratory Visualisations

Corpus refinement counts and class distribution for the thesis report.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Corpus refinement
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=['Raw Scraped', 'Deduplicated (Final)'], y=[14457, len(df)],
            palette='viridis', ax=ax)
ax.set_title('Corpus Refinement: Raw → Unique Tweets')
ax.set_ylabel('Number of Tweets')
plt.tight_layout(); plt.show()

# 2. 5-class distribution
fig, ax = plt.subplots(figsize=(9, 5))
dist = df['label'].value_counts().sort_index().rename(id2label)
dist.plot(kind='barh', color='teal', ax=ax)
ax.set_title('Frequency of Malaria Discussion Categories (5-class)')
ax.set_xlabel('Count')
plt.tight_layout(); plt.show()

---
## Phase 2: Streamlit Dashboard — The "Interface"

Writes `app.py` with:
- Single-tweet classification + confidence bar for all 5 classes  
- CSV batch inference with downloadable output  
- Plotly distribution chart with Misinformation highlighted  
- Confidence threshold slider

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import pandas as pd
import plotly.express as px
import torch

st.set_page_config(page_title="Malaria Discourse Classifier", page_icon="🦟", layout="wide")
st.title("🦟 Malaria Discourse Classifier (Nigeria)")
st.markdown("Classifies tweets into **5 categories** using a fine-tuned **BERTweet** model.")

MODEL_PATH = "./saved_bertweet"
LABELS = ["Symptoms & Burden", "Treatment & Health System",
          "Prevention & Awareness", "Misinformation", "Irrelevant"]
LABEL_COLORS = {
    "Symptoms & Burden":         "#4C72B0",
    "Treatment & Health System": "#55A868",
    "Prevention & Awareness":    "#C44E52",
    "Misinformation":            "#DD8452",   # highlighted
    "Irrelevant":                "#8172B2",
}

@st.cache_resource
def load_model():
    tok = AutoTokenizer.from_pretrained(MODEL_PATH, normalization=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    return tok, mdl

tokenizer, model = load_model()

def predict(text: str):
    import re
    text = re.sub(r'http\S+', 'HTTPURL', text)
    text = re.sub(r'@\w+', '@USER', text)
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1).squeeze().tolist()
    pred  = int(torch.argmax(logits))
    return LABELS[pred], probs

# ── Tabs ────────────────────────────────────────────────────────────────────
tab1, tab2 = st.tabs(["Single Tweet", "Batch CSV"])

with tab1:
    user_input = st.text_area("Enter a tweet:", "I have fever and chills. Is it malaria?", height=100)
    if st.button("Classify"):
        label, probs = predict(user_input)
        st.success(f"**Predicted:** {label}")
        conf_df = pd.DataFrame({"Category": LABELS, "Confidence": probs})
        fig = px.bar(conf_df, x="Confidence", y="Category", orientation="h",
                     color="Category", color_discrete_map=LABEL_COLORS,
                     title="Confidence Scores")
        fig.update_layout(showlegend=False, xaxis_tickformat=".0%")
        st.plotly_chart(fig, use_container_width=True)

with tab2:
    threshold = st.slider("Confidence threshold", 0.0, 1.0, 0.5, 0.05)
    uploaded = st.file_uploader("Upload CSV with a 'text' column", type="csv")
    if uploaded:
        batch_df = pd.read_csv(uploaded)
        labels, confs = [], []
        for txt in batch_df['text']:
            lbl, probs = predict(str(txt))
            conf = max(probs)
            labels.append(lbl if conf >= threshold else "Low Confidence")
            confs.append(round(conf, 4))
        batch_df['predicted_label'] = labels
        batch_df['confidence']      = confs
        st.dataframe(batch_df[['text', 'predicted_label', 'confidence']].head(20))

        # Distribution chart
        dist = batch_df['predicted_label'].value_counts().reset_index()
        dist.columns = ['Category', 'Count']
        fig2 = px.pie(dist, names='Category', values='Count',
                      color='Category', color_discrete_map=LABEL_COLORS,
                      title="Category Distribution")
        st.plotly_chart(fig2, use_container_width=True)

        csv_out = batch_df.to_csv(index=False).encode()
        st.download_button("⬇ Download Results CSV", csv_out,
                           "classified_tweets.csv", "text/csv")

st.sidebar.info("BERTweet fine-tuned on Nigerian malaria tweets.\nFinal Year Project — Infodemiology.")


---
## Phase 3: Real-Time Geospatial Surveillance — The "Infoveillance"

Operationalises the Infodemiology framework from Section 2.2.3.
- Fetches live tweets via `ntscraper` using malaria seed terms filtered to Nigeria  
- Runs BERTweet inference inline  
- Renders a Folium map: 🔴 Symptoms & Burden | 🟠 Misinformation hotspots  
- Flags Misinformation predictions in a separate alert table

In [ ]:
from ntscraper import Nitter
import pandas as pd, re, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Load saved model ─────────────────────────────────────────────────────────
MODEL_PATH = "/content/drive/MyDrive/NLP_twitter/saved_bertweet"
tok = AutoTokenizer.from_pretrained(MODEL_PATH, normalization=True)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
mdl.eval()

LABELS = ["Symptoms & Burden", "Treatment & Health System",
          "Prevention & Awareness", "Misinformation", "Irrelevant"]

SEED_TERMS = ["malaria Nigeria", "#EndMalaria", "artemisinin Nigeria",
              "coartem", "fever chills Nigeria", "paludisme Nigeria"]

def preprocess(t):
    t = re.sub(r'http\S+', 'HTTPURL', str(t))
    return re.sub(r'@\w+', '@USER', t)

def classify(text):
    inputs = tok(preprocess(text), return_tensors="pt",
                 truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        probs = torch.softmax(mdl(**inputs).logits, -1).squeeze().tolist()
    return LABELS[int(torch.argmax(torch.tensor(probs)))], max(probs)

# ── Scrape ───────────────────────────────────────────────────────────────────
scraper = Nitter(log_level=1)
rows = []
for term in SEED_TERMS:
    try:
        results = scraper.get_tweets(term, mode='term', number=50)
        for t in results.get('tweets', []):
            rows.append({'text': t['text'],
                         'date': t.get('date',''),
                         'link': t.get('link','')})
    except Exception as e:
        print(f"Skipping '{term}': {e}")

live_df = pd.DataFrame(rows).drop_duplicates('text')
live_df[['predicted_label', 'confidence']] = live_df['text'].apply(
    lambda x: pd.Series(classify(x))
)
print(f"Fetched {len(live_df)} unique tweets.")

# ── Misinformation alert panel ────────────────────────────────────────────────
misinfo = live_df[live_df['predicted_label'] == 'Misinformation']
print(f"\n⚠️  {len(misinfo)} Misinformation tweets flagged:")
print(misinfo[['text', 'confidence']].to_string(index=False))

In [ ]:
# ── Geospatial map (requires location metadata in tweets) ────────────────────
# If tweets lack lat/lon, assign approximate state-level coords from text heuristics.
import folium

NGA_CENTER = [9.0820, 8.6753]
m = folium.Map(location=NGA_CENTER, zoom_start=6, tiles="CartoDB positron")

PIN_COLORS = {
    "Symptoms & Burden":         "red",
    "Treatment & Health System": "green",
    "Prevention & Awareness":    "blue",
    "Misinformation":            "orange",
    "Irrelevant":                "gray",
}

# Placeholder: jitter around Nigeria centroid until real geo data is available
import random, math
random.seed(42)
for _, row in live_df.iterrows():
    lat = NGA_CENTER[0] + random.uniform(-4, 4)
    lon = NGA_CENTER[1] + random.uniform(-6, 6)
    color = PIN_COLORS.get(row['predicted_label'], 'gray')
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{row['predicted_label']}</b><br>{row['text'][:120]}...<br>Conf: {row['confidence']:.2%}",
            max_width=250
        )
    ).add_to(m)

m.save("malaria_surveillance_map.html")
print("Map saved to malaria_surveillance_map.html")
m

---
## Phase 4: Launch Dashboard (Colab)

In [ ]:
import urllib
print("Tunnel IP:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
!streamlit run app.py & npx localtunnel --port 8501